In [28]:
import urllib.parse
import pandas as pd
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

def get_engine(cfg):
    user = cfg["user"]
    pw = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    db = cfg["dbname"]
    return create_engine(
        f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}?connect_timeout=5",
        pool_pre_ping=True
    )

engine = get_engine(DB_CONFIG)

# 최신 end_day의 upper_outlier 1건 사용 (end_day 미지정이므로 운영상 가장 합리적인 기본값)
SQL_BOUNDARY = text("""
SELECT end_day, upper_outlier
FROM e4_predictive_maintenance.non_pd_cal_test_ct_summary
WHERE test_contents = :tc
  AND upper_outlier IS NOT NULL
ORDER BY end_day DESC
LIMIT 1
""")

b = pd.read_sql(SQL_BOUNDARY, engine, params={"tc": "1.32_dark_curr_check"})
if b.empty:
    raise RuntimeError("[ERROR] non_pd_cal_test_ct_summary에서 1.32_dark_curr_check upper_outlier를 찾지 못했습니다.")

boundary_run_time = float(b.loc[0, "upper_outlier"])
boundary_src_day  = str(b.loc[0, "end_day"])

print(f"[OK] boundary_run_time={boundary_run_time} (source end_day={boundary_src_day})")

[OK] boundary_run_time=26.33 (source end_day=2025-11-13)


In [29]:
# 조건
TARGET_END_DAY = "20251219"
TARGET_REMARK  = "Non-PD"

SQL_DATA = text("""
SELECT
    barcode_information,
    remark,
    station,
    end_day,
    end_time,
    run_time
FROM a2_fct_table.fct_table
WHERE end_day = :end_day
  AND remark  = :remark
""")

df = pd.read_sql(
    SQL_DATA,
    engine,
    params={"end_day": TARGET_END_DAY, "remark": TARGET_REMARK}
)

df["run_time"] = pd.to_numeric(df["run_time"], errors="coerce")

# boundary 값 컬럼화
df["boundary_run_time"] = boundary_run_time

# boundary 초과만
df = df[df["run_time"] > df["boundary_run_time"]].copy()

# barcode 1개당 1행
df = (
    df.sort_values(["barcode_information", "end_time"])
      .groupby("barcode_information", as_index=False)
      .agg({
          "remark": "first",
          "station": "first",
          "end_day": "first",
          "end_time": "max",
          "boundary_run_time": "first",
          "run_time": "first"
      })
)

# ============================
# TOP 5% 컷 계산
# ============================
cut = df["run_time"].quantile(0.95)

df_top = df[df["run_time"] >= cut].copy()

df_top = df_top.sort_values("run_time", ascending=False).reset_index(drop=True)

# 출력 컬럼 고정
df_top = df_top[[
    "barcode_information",
    "remark",
    "station",
    "end_day",
    "end_time",
    "boundary_run_time",
    "run_time"
]]

print(f"[OK] boundary={boundary_run_time} / TOP5% cut={cut:.2f} / rows={len(df_top)}")
df_top

[OK] boundary=26.33 / TOP5% cut=33.50 / rows=7


,barcode_information,remark,station,end_day,end_time,boundary_run_time,run_time
0,BA1WJ25353500229UPC3T-14F014-AC,Non-PD,FCT1,20251219,185750,26.33,50.4
1,BA1WJ25353504258UPC3T-14F014-AC,Non-PD,FCT1,20251219,165704,26.33,39.3
2,BA1WJ25353500062UPC3T-14F014-AC,Non-PD,FCT1,20251219,201231,26.33,38.9
3,BA1WJ25353500067UPC3T-14F014-AC,Non-PD,FCT1,20251219,201514,26.33,38.5
4,BA1WJ25353500872UPC3T-14F014-AC,Non-PD,FCT1,20251219,184653,26.33,38.5
5,BA1WJ25353504500UPC3T-14F014-AC,Non-PD,FCT1,20251219,162309,26.33,34.6
6,BA1WJ25353500841UPC3T-14F014-AC,Non-PD,FCT1,20251219,183959,26.33,34.4


In [30]:
# ============================================
# Cell X0) 공통 유틸: run_time 재배치 + 300행 스크롤(상하/좌우) 출력
# ============================================

import pandas as pd
from IPython.display import display, HTML

def _reorder_run_time_after_end_time(df: pd.DataFrame,
                                    run_time_col: str = "run_time",
                                    after_col: str = "end_time") -> pd.DataFrame:
    if df is None or len(df) == 0:
        return df
    cols = list(df.columns)
    if run_time_col not in cols:
        return df
    if after_col not in cols:
        cols.remove(run_time_col)
        cols.insert(0, run_time_col)
        return df[cols]
    cols.remove(run_time_col)
    idx = cols.index(after_col) + 1
    cols.insert(idx, run_time_col)
    return df[cols]

def display_scroll_df(df: pd.DataFrame, n: int = 300, height_px: int = 560,
                      float_format: dict | None = None, title: str | None = None):
    if df is None:
        raise ValueError("[ERROR] df가 None 입니다.")
    view = df.head(n).copy()

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)

    sty = view.style
    if float_format:
        sty = sty.format(float_format, na_rep="")

    html = sty.to_html()
    wrapper = f"""
    <div style="border:1px solid #ddd; padding:8px;">
      {f"<div style='font-weight:600; margin-bottom:6px;'>{title}</div>" if title else ""}
      <div style="overflow:auto; width:100%; max-height:{int(height_px)}px;">
        {html}
      </div>
    </div>
    """
    display(HTML(wrapper))

print("[OK] Cell X0 loaded")

[OK] Cell X0 loaded


In [31]:
# ============================================
# Cell A) boundary_run_time 로드 (당신이 올린 1st cell 유지)
# ============================================

import urllib.parse
import pandas as pd
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

def get_engine(cfg):
    user = cfg["user"]
    pw = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    db = cfg["dbname"]
    return create_engine(
        f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}?connect_timeout=5",
        pool_pre_ping=True
    )

engine = get_engine(DB_CONFIG)

SQL_BOUNDARY = text("""
SELECT end_day, upper_outlier
FROM e4_predictive_maintenance.non_pd_cal_test_ct_summary
WHERE test_contents = :tc
  AND upper_outlier IS NOT NULL
ORDER BY end_day DESC
LIMIT 1
""")

b = pd.read_sql(SQL_BOUNDARY, engine, params={"tc": "1.32_dark_curr_check"})
if b.empty:
    raise RuntimeError("[ERROR] non_pd_cal_test_ct_summary에서 1.32_dark_curr_check upper_outlier를 찾지 못했습니다.")

boundary_run_time = float(b.loc[0, "upper_outlier"])
boundary_src_day  = str(b.loc[0, "end_day"])

print(f"[OK] boundary_run_time={boundary_run_time} (source end_day={boundary_src_day})")

[OK] boundary_run_time=26.33 (source end_day=2025-11-13)


In [32]:
# ============================================
# Cell B) TOP 5% 산출 (당신이 올린 2nd cell 유지 + 출력 스크롤만 적용)
#  - 결과: df_top (run_time 포함)
# ============================================

from sqlalchemy import text
import pandas as pd

TARGET_END_DAY = "20251219"
TARGET_REMARK  = "Non-PD"

SQL_DATA = text("""
SELECT
    barcode_information,
    remark,
    station,
    end_day,
    end_time,
    run_time
FROM a2_fct_table.fct_table
WHERE end_day = :end_day
  AND remark  = :remark
""")

df = pd.read_sql(SQL_DATA, engine, params={"end_day": TARGET_END_DAY, "remark": TARGET_REMARK})
df["run_time"] = pd.to_numeric(df["run_time"], errors="coerce")

df["boundary_run_time"] = boundary_run_time
df = df[df["run_time"] > df["boundary_run_time"]].copy()

df = (
    df.sort_values(["barcode_information", "end_time"])
      .groupby("barcode_information", as_index=False)
      .agg({
          "remark": "first",
          "station": "first",
          "end_day": "first",
          "end_time": "max",
          "boundary_run_time": "first",
          "run_time": "first"
      })
)

cut = df["run_time"].quantile(0.95)

df_top = df[df["run_time"] >= cut].copy()
df_top = df_top.sort_values("run_time", ascending=False).reset_index(drop=True)

df_top = df_top[[
    "barcode_information",
    "remark",
    "station",
    "end_day",
    "end_time",
    "boundary_run_time",
    "run_time"
]]

print(f"[OK] boundary={boundary_run_time} / TOP5% cut={cut:.2f} / rows={len(df_top)}")

df_top = _reorder_run_time_after_end_time(df_top, "run_time", "end_time")
display_scroll_df(
    df_top, n=300, height_px=520,
    float_format={"boundary_run_time":"{:.2f}", "run_time":"{:.2f}"},
    title="df_top (TOP 5%, top 300, scroll)"
)

[OK] boundary=26.33 / TOP5% cut=33.50 / rows=7


,barcode_information,remark,station,end_day,end_time,run_time,boundary_run_time
0,BA1WJ25353500229UPC3T-14F014-AC,Non-PD,FCT1,20251219,185750,50.40,26.33
1,BA1WJ25353504258UPC3T-14F014-AC,Non-PD,FCT1,20251219,165704,39.30,26.33
2,BA1WJ25353500062UPC3T-14F014-AC,Non-PD,FCT1,20251219,201231,38.90,26.33
3,BA1WJ25353500067UPC3T-14F014-AC,Non-PD,FCT1,20251219,201514,38.50,26.33
4,BA1WJ25353500872UPC3T-14F014-AC,Non-PD,FCT1,20251219,184653,38.50,26.33
5,BA1WJ25353504500UPC3T-14F014-AC,Non-PD,FCT1,20251219,162309,34.60,26.33
6,BA1WJ25353500841UPC3T-14F014-AC,Non-PD,FCT1,20251219,183959,34.40,26.33


In [33]:
# ============================================
# Cell X1) df_top 바코드 → fct_detail 조회 + station/run_time/boundary_run_time merge
#  - 핵심: df_top의 run_time을 그대로 가져와서 df_detail에 붙임
# ============================================

import pandas as pd
from sqlalchemy import text

if "df_top" not in globals() or len(df_top) == 0:
    raise ValueError("[ERROR] df_top이 없거나 비어있습니다. Cell B를 먼저 실행하세요.")

# barcode list
barcodes = (
    df_top["barcode_information"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)
print(f"[OK] Top barcodes = {len(barcodes)}")

# df_top end_day는 TEXT(YYYYMMDD) → fct_detail DATE(YYYY-MM-DD)
TARGET_END_DAY_TEXT = str(df_top["end_day"].iloc[0]).strip()
TARGET_END_DAY_DATE = pd.to_datetime(TARGET_END_DAY_TEXT, format="%Y%m%d", errors="raise").strftime("%Y-%m-%d")
TARGET_REMARK = str(df_top["remark"].iloc[0]).strip() if "remark" in df_top.columns else "PD"

print(f"[OK] TARGET_END_DAY_TEXT={TARGET_END_DAY_TEXT} / TARGET_END_DAY_DATE={TARGET_END_DAY_DATE} / REMARK={TARGET_REMARK}")

SQL_FCT_DETAIL = text("""
SELECT
    barcode_information,
    remark,
    end_day,
    end_time,
    contents,
    test_ct,
    test_time
FROM c1_fct_detail.fct_detail
WHERE end_day = CAST(:end_day AS date)
  AND remark = :remark
  AND barcode_information = ANY(CAST(:barcodes AS text[]))
""")

df_detail = pd.read_sql(
    SQL_FCT_DETAIL,
    engine,
    params={"end_day": TARGET_END_DAY_DATE, "remark": TARGET_REMARK, "barcodes": barcodes}
)

# ✅ df_top에서 station/run_time/boundary_run_time 붙이기(barcode 기준 1행)
df_meta = df_top[["barcode_information", "station", "run_time", "boundary_run_time"]].copy()
df_meta["barcode_information"] = df_meta["barcode_information"].astype(str)
df_meta["station"] = df_meta["station"].astype(str)
df_meta["run_time"] = pd.to_numeric(df_meta["run_time"], errors="coerce")
df_meta["boundary_run_time"] = pd.to_numeric(df_meta["boundary_run_time"], errors="coerce")
df_meta = df_meta.drop_duplicates("barcode_information")

df_detail["barcode_information"] = df_detail["barcode_information"].astype(str)
df_detail = df_detail.merge(df_meta, on="barcode_information", how="left")

# 컬럼 순서(요구: barcode 다음 station, 그리고 end_time 다음 run_time)
df_detail = df_detail[[
    "barcode_information",
    "station",
    "remark",
    "end_day",
    "end_time",
    "run_time",
    "boundary_run_time",
    "contents",
    "test_ct",
    "test_time"
]].copy()

print(f"[OK] df_detail rows={len(df_detail)}")

df_detail = _reorder_run_time_after_end_time(df_detail, "run_time", "end_time")
display_scroll_df(
    df_detail, n=300, height_px=560,
    float_format={"run_time":"{:.2f}", "boundary_run_time":"{:.2f}", "test_ct":"{:.3f}"},
    title="df_detail (top 300, scroll)"
)

[OK] Top barcodes = 7
[OK] TARGET_END_DAY_TEXT=20251219 / TARGET_END_DAY_DATE=2025-12-19 / REMARK=Non-PD
[OK] df_detail rows=1497


,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,contents,test_ct,test_time
0,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,3~192.168.108.151~FCT~PPIDBA1WJ25353504258UPC3T-14F014-AC~END,,16:56:24.22
1,BA1WJ25353504500UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:23:10,34.60,26.33,3~192.168.108.151~FCT~PPIDBA1WJ25353504500UPC3T-14F014-AC~END,,16:22:34.75
2,BA1WJ25353504500UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:23:10,34.60,26.33,00674~192.168.108.151~FCT~PPIDBA1WJ25353504500UPC3T-14F014-AC~00~OK~END,0.020,16:22:34.77
3,BA1WJ25353504500UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:23:10,34.60,26.33,START :: DO SET,2.250,16:22:37.02
4,BA1WJ25353504500UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:23:10,34.60,26.33,DO_SET Value :: 0900800080001000,0.040,16:22:37.06
5,BA1WJ25353504500UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:23:10,34.60,26.33,테스트 결과 : OK,0.020,16:22:37.08
6,BA1WJ25353504500UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:23:10,34.60,26.33,START :: LOAD_A SET CC MODE,0.010,16:22:37.09
7,BA1WJ25353504500UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:23:10,34.60,26.33,Return Value :: OK,0.020,16:22:37.11
8,BA1WJ25353504500UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:23:10,34.60,26.33,테스트 결과 : OK,0.020,16:22:37.13
9,BA1WJ25353504500UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:23:10,34.60,26.33,START :: LOAD_A SET RANG,0.020,16:22:37.15


In [34]:
# ============================================
# Cell X2) group 생성 + 정렬 + group 제외 규칙 적용
#  - group_key: barcode_information + end_day + end_time (station 제외)
# ============================================

import pandas as pd

df2 = df_detail.copy()

df2["barcode_information"] = df2["barcode_information"].astype(str)
df2["station"] = df2["station"].astype(str)
df2["remark"] = df2["remark"].astype(str)
df2["end_day"] = pd.to_datetime(df2["end_day"], errors="coerce").dt.strftime("%Y-%m-%d")
df2["end_time"] = df2["end_time"].astype(str)

def parse_test_time_to_ts(end_day_str, t):
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return pd.NaT
    s = str(t).strip()
    if s == "" or s.lower() == "none":
        return pd.NaT
    return pd.to_datetime(f"{end_day_str} {s}", errors="coerce")

df2["_test_ts"] = [parse_test_time_to_ts(d, t) for d, t in zip(df2["end_day"], df2["test_time"])]

df2["group_key"] = df2["barcode_information"] + "|" + df2["end_day"] + "|" + df2["end_time"]
df2["group"] = pd.factorize(df2["group_key"], sort=False)[0] + 1

# 제외 (6)
skip_mask = df2["contents"].astype(str).str.contains("START :: MES 이전공정 체크 SKIP", na=False)
skip_groups = set(df2.loc[skip_mask, "group"].unique().tolist())

# (7)(8)
df2_sorted_tmp = df2.sort_values(["group", "_test_ts"], ascending=[True, True]).copy()

def first_3_row(sub):
    m = sub["contents"].astype(str).str.startswith("3~", na=False)
    if not m.any():
        return None
    return sub.loc[m].iloc[0]

valid_groups = []
for g, sub in df2_sorted_tmp.groupby("group", sort=False):
    if g in skip_groups:
        continue
    r = first_3_row(sub)
    if r is None:
        continue
    if pd.isna(r["test_ct"]):
        valid_groups.append(g)

df2 = df2[df2["group"].isin(valid_groups)].copy()

df2["_is_first_3_null"] = (
    df2["contents"].astype(str).str.startswith("3~", na=False) &
    df2["test_ct"].isna()
).astype(int)

df2 = df2.sort_values(["group", "_is_first_3_null", "_test_ts"], ascending=[True, False, True]).reset_index(drop=True)
df2.drop(columns=["group_key"], inplace=True, errors="ignore")

print(f"[OK] df2 rows={len(df2)} / groups={df2['group'].nunique()}")

df2 = _reorder_run_time_after_end_time(df2, "run_time", "end_time")
display_scroll_df(
    df2, n=300, height_px=560,
    float_format={"run_time":"{:.2f}", "boundary_run_time":"{:.2f}", "test_ct":"{:.3f}"},
    title="df2 (top 300, scroll)"
)

[OK] df2 rows=1497 / groups=7


,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,contents,test_ct,test_time,_test_ts,group,_is_first_3_null
0,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,3~192.168.108.151~FCT~PPIDBA1WJ25353504258UPC3T-14F014-AC~END,,16:56:24.22,2025-12-19 16:56:24.220000,1,1
1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,00674~192.168.108.151~FCT~PPIDBA1WJ25353504258UPC3T-14F014-AC~00~OK~END,0.020,16:56:24.24,2025-12-19 16:56:24.240000,1,0
2,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,START :: DO SET,2.350,16:56:26.59,2025-12-19 16:56:26.590000,1,0
3,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,DO_SET Value :: 0900800080001000,0.040,16:56:26.63,2025-12-19 16:56:26.630000,1,0
4,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,테스트 결과 : OK,0.010,16:56:26.64,2025-12-19 16:56:26.640000,1,0
5,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,START :: LOAD_A SET CC MODE,0.020,16:56:26.66,2025-12-19 16:56:26.660000,1,0
6,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,Return Value :: OK,0.010,16:56:26.67,2025-12-19 16:56:26.670000,1,0
7,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,테스트 결과 : OK,0.010,16:56:26.68,2025-12-19 16:56:26.680000,1,0
8,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,START :: LOAD_A SET RANG,0.020,16:56:26.70,2025-12-19 16:56:26.700000,1,0
9,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,Return Value :: OK,0.110,16:56:26.81,2025-12-19 16:56:26.810000,1,0


In [35]:
# ============================================
# Cell X3) from_to_test_ct 생성 (OK/NG만)
# ============================================

import pandas as pd

df3 = df2.copy()
df3["_base_ts"] = df3.groupby("group")["_test_ts"].transform("first")

is_okng = df3["contents"].astype(str).isin(["테스트 결과 : OK", "테스트 결과 : NG"])

df3["from_to_test_ct"] = pd.NA
mask = is_okng & df3["_test_ts"].notna() & df3["_base_ts"].notna()
df3.loc[mask, "from_to_test_ct"] = (df3.loc[mask, "_test_ts"] - df3.loc[mask, "_base_ts"]).dt.total_seconds()

print(f"[OK] from_to_test_ct filled rows={df3['from_to_test_ct'].notna().sum()}")

df3 = _reorder_run_time_after_end_time(df3, "run_time", "end_time")
display_scroll_df(
    df3, n=300, height_px=560,
    float_format={"run_time":"{:.2f}", "boundary_run_time":"{:.2f}", "from_to_test_ct":"{:.3f}"},
    title="df3 (top 300, scroll)"
)

[OK] from_to_test_ct filled rows=469


,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,contents,test_ct,test_time,_test_ts,group,_is_first_3_null,_base_ts,from_to_test_ct
0,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,3~192.168.108.151~FCT~PPIDBA1WJ25353504258UPC3T-14F014-AC~END,,16:56:24.22,2025-12-19 16:56:24.220000,1,1,2025-12-19 16:56:24.220000,
1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,00674~192.168.108.151~FCT~PPIDBA1WJ25353504258UPC3T-14F014-AC~00~OK~END,0.020000,16:56:24.24,2025-12-19 16:56:24.240000,1,0,2025-12-19 16:56:24.220000,
2,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,START :: DO SET,2.350000,16:56:26.59,2025-12-19 16:56:26.590000,1,0,2025-12-19 16:56:24.220000,
3,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,DO_SET Value :: 0900800080001000,0.040000,16:56:26.63,2025-12-19 16:56:26.630000,1,0,2025-12-19 16:56:24.220000,
4,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,테스트 결과 : OK,0.010000,16:56:26.64,2025-12-19 16:56:26.640000,1,0,2025-12-19 16:56:24.220000,2.420
5,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,START :: LOAD_A SET CC MODE,0.020000,16:56:26.66,2025-12-19 16:56:26.660000,1,0,2025-12-19 16:56:24.220000,
6,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,Return Value :: OK,0.010000,16:56:26.67,2025-12-19 16:56:26.670000,1,0,2025-12-19 16:56:24.220000,
7,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,테스트 결과 : OK,0.010000,16:56:26.68,2025-12-19 16:56:26.680000,1,0,2025-12-19 16:56:24.220000,2.460
8,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,START :: LOAD_A SET RANG,0.020000,16:56:26.70,2025-12-19 16:56:26.700000,1,0,2025-12-19 16:56:24.220000,
9,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,Return Value :: OK,0.110000,16:56:26.81,2025-12-19 16:56:26.810000,1,0,2025-12-19 16:56:24.220000,


In [36]:
# ============================================
# Cell X4) okng_seq + test_contents 매핑 + df_final 출력
# ============================================

import pandas as pd

df4 = df3.copy()

MAP_1_67 = {
    1:"0.00_d_sig_val_090_set", 2:"0.00_load_a_cc_set", 3:"0.00_dmm_c_rng_set", 4:"0.00_load_c_cc_set",
    5:"0.00_dmm_c_rng_set", 6:"0.00_dmm_regi_set", 7:"0.00_dmm_regi_ac_0.6_set",
    8:"1.00_mini_b_short_check", 9:"1.01_usb_a_short_check", 10:"1.02_usb_c_short_check",
    11:"1.03_d_sig_val_000_set", 12:"1.03_dmm_regi_set", 13:"1.03_dmm_regi_ac_0.6_set",
    14:"1.03_pin12_short_check", 15:"1.04_pin23_short_check", 16:"1.05_pin34_short_check",
    17:"1.06_dmm_dc_v_set", 18:"1.06_dmm_ac_0.6_set", 19:"1.06_dmm_c_set",
    20:"1.06_load_a_sensing_on", 21:"1.06_load_c_sensing_on",
    22:"1.06_ps_16.5v_set", 23:"1.06_ps_on", 24:"1.06_dmm_ac_0.6_set", 25:"1.06_input_16.5v",
    26:"1.07_idle_c_check", 27:"1.08_fw_ver_check", 28:"1.09_chip_id_check",
    29:"1.10_dmm_3c_rng_set", 30:"1.10_load_a_5.5c_set", 31:"1.10_load_a_on", 32:"1.10_usb_a_v_check",
    33:"1.11_usb_a_c_check", 34:"1.12_usb_c_v_check", 35:"1.13_load_a_off", 36:"1.13_load_c_5.5c_set",
    37:"1.13_load_c_on", 38:"1.13_overcurr_usb_c_v", 39:"1.14_overcurr_usb_c_c", 40:"1.15_usb_a_v",
    41:"1.16_load_c_off", 42:"1.16_dut_reset", 43:"1.16_cc_aside_check", 44:"1.17_cc_bside_check",
    45:"1.18_load_a_2.4c_set", 46:"1.18_load_c_3c_set", 47:"1.18_load_a_on", 48:"1.18_load_c_on",
    49:"1.18_usb_a_v_check", 50:"1.19_usb_a_c_check", 51:"1.20_usb_c_v_check", 52:"1.21_usb_c_c_check",
    53:"1.22_load_a_off", 54:"1.22_load_c_off", 55:"1.22_usb_c_carplay", 56:"1.23_usb_a_carplay",
    57:"1.24_usb_c_pm1", 58:"1.25_usb_c_pm2", 59:"1.26_usb_c_pm3", 60:"1.27_usb_c_pm4",
    61:"1.28_usb_a_pm1", 62:"1.29_usb_a_pm2", 63:"1.30_usb_a_pm3", 64:"1.31_usb_a_pm4",
    65:"1.32_dmm_ac_0.6_set", 66:"1.32_dmm_c_rng_set", 67:"1.32_dark_curr_check"
}

is_okng = df4["contents"].astype(str).isin(["테스트 결과 : OK", "테스트 결과 : NG"])

df4["okng_seq"] = pd.NA
df4.loc[is_okng, "okng_seq"] = (
    df4.loc[is_okng]
       .groupby("group")
       .cumcount()
       .add(1)
       .astype(int)
)

df4["test_contents"] = pd.NA
df4.loc[is_okng, "test_contents"] = df4.loc[is_okng, "okng_seq"].map(MAP_1_67)

df_final = df4[[
    "group",
    "barcode_information",
    "station",
    "remark",
    "end_day",
    "end_time",
    "run_time",
    "boundary_run_time",
    "test_time",
    "contents",
    "test_contents",
    "test_ct",
    "from_to_test_ct",
    "okng_seq"
]].copy()

print(f"[OK] df_final rows={len(df_final)} / groups={df_final['group'].nunique()}")

df_final = _reorder_run_time_after_end_time(df_final, "run_time", "end_time")
display_scroll_df(
    df_final, n=300, height_px=560,
    float_format={"run_time":"{:.2f}", "boundary_run_time":"{:.2f}", "test_ct":"{:.3f}", "from_to_test_ct":"{:.3f}"},
    title="df_final (top 300, scroll)"
)

[OK] df_final rows=1497 / groups=7


,group,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,test_time,contents,test_contents,test_ct,from_to_test_ct,okng_seq
0,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:24.22,3~192.168.108.151~FCT~PPIDBA1WJ25353504258UPC3T-14F014-AC~END,,,,
1,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:24.24,00674~192.168.108.151~FCT~PPIDBA1WJ25353504258UPC3T-14F014-AC~00~OK~END,,0.020,,
2,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:26.59,START :: DO SET,,2.350,,
3,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:26.63,DO_SET Value :: 0900800080001000,,0.040,,
4,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:26.64,테스트 결과 : OK,0.00_d_sig_val_090_set,0.010,2.420,1
5,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:26.66,START :: LOAD_A SET CC MODE,,0.020,,
6,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:26.67,Return Value :: OK,,0.010,,
7,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:26.68,테스트 결과 : OK,0.00_load_a_cc_set,0.010,2.460,2
8,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:26.70,START :: LOAD_A SET RANG,,0.020,,
9,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:26.81,Return Value :: OK,,0.110,,


In [37]:
# ============================================
# Cell X5) df15 생성 (contents/test_ct 제외, test_contents NA 제외)
# ============================================

import pandas as pd

df15 = df_final.dropna(subset=["test_contents"]).copy()

df15 = df15[[
    "group",
    "barcode_information",
    "station",
    "remark",
    "end_day",
    "end_time",
    "run_time",
    "boundary_run_time",
    "test_time",
    "test_contents",
    "from_to_test_ct",
    "okng_seq"
]].reset_index(drop=True)

print(f"[OK] df15 rows={len(df15)} / groups={df15['group'].nunique()}")

df15 = _reorder_run_time_after_end_time(df15, "run_time", "end_time")
display_scroll_df(
    df15, n=300, height_px=560,
    float_format={"run_time":"{:.2f}", "boundary_run_time":"{:.2f}", "from_to_test_ct":"{:.3f}"},
    title="df15 (top 300, scroll)"
)

[OK] df15 rows=469 / groups=7


,group,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,test_time,test_contents,from_to_test_ct,okng_seq
0,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:26.64,0.00_d_sig_val_090_set,2.420,1
1,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:26.68,0.00_load_a_cc_set,2.460,2
2,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:26.83,0.00_dmm_c_rng_set,2.610,3
3,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:26.87,0.00_load_c_cc_set,2.650,4
4,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:27.09,0.00_dmm_c_rng_set,2.870,5
5,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:27.32,0.00_dmm_regi_set,3.100,6
6,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:27.67,0.00_dmm_regi_ac_0.6_set,3.450,7
7,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:27.75,1.00_mini_b_short_check,3.530,8
8,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:28.00,1.01_usb_a_short_check,3.780,9
9,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,16:56:28.28,1.02_usb_c_short_check,4.060,10


In [38]:
# ============================================
# Cell X6) df15에 summary upper_outlier + problem1~4 JOIN (Non-PD Summary)
# 1) upper_outlier -> boundary_test_ct 로 컬럼명 변경
# 2) 출력 컬럼 순서 고정:
#    group, barcode_information, station, remark, end_day, end_time, run_time,
#    boundary_run_time, okng_seq, test_contents, test_time, from_to_test_ct,
#    boundary_test_ct, problem1, problem2, problem3, problem4
#  - 출력: 300행, 상하/좌우 스크롤
# ============================================

import pandas as pd
from sqlalchemy import text

if "df15" not in globals() or len(df15) == 0:
    raise ValueError("[ERROR] df15가 없거나 비어있습니다. Cell X5를 먼저 실행하세요.")

TARGET_END_DAY = str(df15["end_day"].dropna().iloc[0]).strip()

# problem1~4 존재 확인
SQL_PROBLEM_COLS = text("""
SELECT column_name
FROM information_schema.columns
WHERE table_schema = 'e4_predictive_maintenance'
  AND table_name   = 'non_pd_cal_test_ct_summary'
  AND column_name IN ('problem1','problem2','problem3','problem4')
ORDER BY column_name
""")
present_problem_cols = pd.read_sql(SQL_PROBLEM_COLS, engine)["column_name"].tolist()
present_problem_cols = [c.strip() for c in present_problem_cols if isinstance(c, str)]

want_problem_cols = ["problem1","problem2","problem3","problem4"]
use_problem_cols  = [c for c in want_problem_cols if c in present_problem_cols]

# end_day 컬럼이 summary에 존재할 수도/없을 수도 있어 안전하게 처리
SQL_HAS_ENDDAY_COL = text("""
SELECT COUNT(*) AS n
FROM information_schema.columns
WHERE table_schema='e4_predictive_maintenance'
  AND table_name='non_pd_cal_test_ct_summary'
  AND column_name='end_day'
""")
has_endday = int(pd.read_sql(SQL_HAS_ENDDAY_COL, engine).iloc[0]["n"]) > 0

# boundary + problem1~4 로드
select_cols = ["test_contents", "upper_outlier"] + use_problem_cols

if has_endday:
    SQL_BOUNDARY = text(f"""
    SELECT {", ".join(select_cols)}
    FROM e4_predictive_maintenance.non_pd_cal_test_ct_summary
    WHERE end_day = :end_day
    """)
    df_boundary = pd.read_sql(SQL_BOUNDARY, engine, params={"end_day": TARGET_END_DAY})

    # 만약 해당 end_day가 비어있으면 최신 end_day로 폴백
    if len(df_boundary) == 0:
        SQL_LATEST_DAY = text("""
        SELECT MAX(end_day) AS max_day
        FROM e4_predictive_maintenance.non_pd_cal_test_ct_summary
        """)
        latest = pd.read_sql(SQL_LATEST_DAY, engine).iloc[0]["max_day"]
        if pd.isna(latest):
            raise ValueError("[ERROR] non_pd_cal_test_ct_summary 테이블이 비어있습니다.")
        latest = str(latest).strip()

        df_boundary = pd.read_sql(SQL_BOUNDARY, engine, params={"end_day": latest})
else:
    # end_day가 없는 테이블이면 전체를 로드해서 test_contents 기준으로만 매칭
    SQL_BOUNDARY = text(f"""
    SELECT {", ".join(select_cols)}
    FROM e4_predictive_maintenance.non_pd_cal_test_ct_summary
    """)
    df_boundary = pd.read_sql(SQL_BOUNDARY, engine)

# 타입/공백 정리
df_boundary["test_contents"] = df_boundary["test_contents"].astype(str).str.strip()
df_boundary["upper_outlier"] = pd.to_numeric(df_boundary["upper_outlier"], errors="coerce").round(2)

for c in use_problem_cols:
    df_boundary[c] = df_boundary[c].astype(str).str.strip()

# JOIN
df15_out = df15.copy()
df15_out["test_contents"] = df15_out["test_contents"].astype(str).str.strip()

join_cols = ["test_contents", "upper_outlier"] + use_problem_cols
df15_out = df15_out.merge(df_boundary[join_cols], on="test_contents", how="left")

# 컬럼명 변경: upper_outlier -> boundary_test_ct
df15_out = df15_out.rename(columns={"upper_outlier": "boundary_test_ct"})

# problem1~4 컬럼 보장(없으면 생성)
for c in want_problem_cols:
    if c not in df15_out.columns:
        df15_out[c] = pd.NA

# 숫자 반올림(요청: 초/수치 2자리)  <-- 원본 df15_out의 값 자체는 2자리로 유지
for c in ["run_time","boundary_run_time","from_to_test_ct","boundary_test_ct"]:
    if c in df15_out.columns:
        df15_out[c] = pd.to_numeric(df15_out[c], errors="coerce").round(2)

# 출력 컬럼 순서 고정
OUT_COLS_X6 = [
    "group","barcode_information","station","remark","end_day","end_time","run_time",
    "boundary_run_time","okng_seq","test_contents","test_time","from_to_test_ct",
    "boundary_test_ct","problem1","problem2","problem3","problem4"
]

missing = [c for c in OUT_COLS_X6 if c not in df15_out.columns]
if missing:
    raise KeyError(f"[ERROR] df15_out에 필요한 컬럼이 없습니다: {missing}")

df15_out = df15_out[OUT_COLS_X6].copy()

print(f"[OK] df15_out rows={len(df15_out)} / boundary_test_ct filled={df15_out['boundary_test_ct'].notna().sum()}")

# =========================================================
# ✅✅✅ 여기만 추가: "표시 전용" df15_out_show 만들어 2자리 고정 출력
# - 원본 df15_out는 그대로 숫자(계산용) 유지
# - display_scroll_df가 포맷 무시해도 문자열이면 강제 2자리로 보임
# =========================================================
df15_out_show = df15_out.copy()
for c in ["run_time","boundary_run_time","from_to_test_ct","boundary_test_ct"]:
    if c in df15_out_show.columns:
        df15_out_show[c] = pd.to_numeric(df15_out_show[c], errors="coerce").map(
            lambda x: "" if pd.isna(x) else f"{x:.2f}"
        )

display_scroll_df(
    df15_out_show,   # ✅ 출력은 show로만 (원본은 안 건드림)
    n=300,
    height_px=560,
    title="Cell X6: df15_out (top 300, scroll)"
)

[OK] df15_out rows=497 / boundary_test_ct filled=497


,group,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,okng_seq,test_contents,test_time,from_to_test_ct,boundary_test_ct,problem1,problem2,problem3,problem4
0,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,1,0.00_d_sig_val_090_set,16:56:26.64,2.42,2.58,relay_board,None,None,None
1,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,2,0.00_load_a_cc_set,16:56:26.68,2.46,2.77,load_a,relay_board,None,None
2,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,3,0.00_dmm_c_rng_set,16:56:26.83,2.61,3.34,dmm,relay_board,None,None
3,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,3,0.00_dmm_c_rng_set,16:56:26.83,2.61,3.34,dmm,relay_board,None,None
4,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,4,0.00_load_c_cc_set,16:56:26.87,2.65,3.36,load_c,relay_board,None,None
5,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,5,0.00_dmm_c_rng_set,16:56:27.09,2.87,3.34,dmm,relay_board,None,None
6,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,5,0.00_dmm_c_rng_set,16:56:27.09,2.87,3.34,dmm,relay_board,None,None
7,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,6,0.00_dmm_regi_set,16:56:27.32,3.10,3.87,dmm,relay_board,None,None
8,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,7,0.00_dmm_regi_ac_0.6_set,16:56:27.67,3.45,4.22,dmm,relay_board,None,None
9,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,8,1.00_mini_b_short_check,16:56:27.75,3.53,4.30,mini_b,dmm,relay_board,None


In [39]:
# ============================================
# Cell X7) diff_ct 생성 (수식 수정)
# - diff_ct = from_to_test_ct - boundary_test_ct
# - 컬럼 순서 고정 출력
# ============================================

import pandas as pd

if "df15_out" not in globals() or len(df15_out) == 0:
    raise ValueError("[ERROR] df15_out가 없거나 비어있습니다. Cell X6를 먼저 실행하세요.")

work = df15_out.copy()

work["boundary_test_ct"] = pd.to_numeric(work["boundary_test_ct"], errors="coerce")
work["from_to_test_ct"]  = pd.to_numeric(work["from_to_test_ct"], errors="coerce")

# ✅ 수식 수정
work["diff_ct"] = work["from_to_test_ct"] - work["boundary_test_ct"]

OUT_COLS_X7 = [
    "group","barcode_information","station","remark","end_day","end_time","run_time",
    "boundary_run_time","okng_seq","test_contents","test_time","from_to_test_ct",
    "boundary_test_ct","diff_ct","problem1","problem2","problem3", "problem4"
]
missing = [c for c in OUT_COLS_X7 if c not in work.columns]
if missing:
    raise KeyError(f"[ERROR] X7 출력에 필요한 컬럼이 없습니다: {missing}")

df15_out2 = work[OUT_COLS_X7].copy()

print(f"[OK] df15_out2 rows={len(df15_out2)} / diff_ct non-null={df15_out2['diff_ct'].notna().sum()}")

display_scroll_df(
    df15_out2,
    n=300,
    height_px=560,
    float_format={
        "run_time":"{:.2f}",
        "boundary_run_time":"{:.2f}",
        "boundary_test_ct":"{:.2f}",
        "from_to_test_ct":"{:.3f}",
        "diff_ct":"{:.3f}",
    },
    title="Cell X7: df15_out2 (diff_ct = from_to_test_ct - boundary_test_ct)"
)

[OK] df15_out2 rows=497 / diff_ct non-null=497


,group,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,okng_seq,test_contents,test_time,from_to_test_ct,boundary_test_ct,diff_ct,problem1,problem2,problem3,problem4
0,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,1,0.00_d_sig_val_090_set,16:56:26.64,2.420,2.58,-0.160,relay_board,None,None,None
1,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,2,0.00_load_a_cc_set,16:56:26.68,2.460,2.77,-0.310,load_a,relay_board,None,None
2,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,3,0.00_dmm_c_rng_set,16:56:26.83,2.610,3.34,-0.730,dmm,relay_board,None,None
3,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,3,0.00_dmm_c_rng_set,16:56:26.83,2.610,3.34,-0.730,dmm,relay_board,None,None
4,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,4,0.00_load_c_cc_set,16:56:26.87,2.650,3.36,-0.710,load_c,relay_board,None,None
5,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,5,0.00_dmm_c_rng_set,16:56:27.09,2.870,3.34,-0.470,dmm,relay_board,None,None
6,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,5,0.00_dmm_c_rng_set,16:56:27.09,2.870,3.34,-0.470,dmm,relay_board,None,None
7,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,6,0.00_dmm_regi_set,16:56:27.32,3.100,3.87,-0.770,dmm,relay_board,None,None
8,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,7,0.00_dmm_regi_ac_0.6_set,16:56:27.67,3.450,4.22,-0.770,dmm,relay_board,None,None
9,1,BA1WJ25353504258UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,16:57:04,39.30,26.33,8,1.00_mini_b_short_check,16:56:27.75,3.530,4.30,-0.770,mini_b,dmm,relay_board,None


In [40]:
# ============================================
# Cell X8) diff_ct TOP 5%만 출력 (run_time 내림차순 정렬)
# - TOP5% 산출 전에 "바코드별 대표행"을 먼저 만든다
#   * 대표행 선정 규칙:
#     1) barcode_information 그룹 내 okng_seq 최소
#     2) (동률이면) diff_ct 최대
#     3) (추가 동률이면) run_time 최대, end_time 빠른 순
# - 대표행들로 diff_ct TOP5% 컷 계산
# ============================================

import pandas as pd

if "df15_out2" not in globals() or len(df15_out2) == 0:
    raise ValueError("[ERROR] df15_out2가 없거나 비어있습니다. Cell X7을 먼저 실행하세요.")

df_spike = df15_out2.copy()

# 숫자형 보정
df_spike["diff_ct"]  = pd.to_numeric(df_spike["diff_ct"], errors="coerce")
df_spike["run_time"] = pd.to_numeric(df_spike["run_time"], errors="coerce")
df_spike["okng_seq"] = pd.to_numeric(df_spike["okng_seq"], errors="coerce")

# 핵심 결측 제거 (대표행 선정/quantile 안정화)
df_spike = df_spike.dropna(subset=["barcode_information", "okng_seq", "diff_ct"]).copy()

# =========================================================
# 1) 바코드별 okng_seq 최소값(min_okng_seq) 계산
# =========================================================
df_spike["min_okng_seq"] = df_spike.groupby("barcode_information")["okng_seq"].transform("min")

# =========================================================
# 2) okng_seq == min_okng_seq 인 행만 남김
#    (여기서부터는 "각 바코드의 가장 빠른 okng_seq 후보군")
# =========================================================
df_minseq = df_spike[df_spike["okng_seq"] == df_spike["min_okng_seq"]].copy()

# =========================================================
# 3) 그 후보군에서 바코드별 대표행 1건 선택
#    - diff_ct 큰 것 우선
#    - 동률이면 run_time 큰 것, end_time 빠른 것
# =========================================================
df_rep = (
    df_minseq.sort_values(
        ["barcode_information", "diff_ct", "run_time", "end_time"],
        ascending=[True, False, False, True]
    )
    .drop_duplicates(subset=["barcode_information"], keep="first")
    .copy()
)

# =========================================================
# 4) 대표행들로 diff_ct TOP5% 컷 산출 후 필터
# =========================================================
cut95 = df_rep["diff_ct"].quantile(0.95)
df_top = df_rep[df_rep["diff_ct"] >= cut95].copy()

# 5) 출력 정렬: run_time 내림차순
df_top = df_top.sort_values("run_time", ascending=False).reset_index(drop=True)

OUT_COLS_X8 = [
    "group","barcode_information","station","remark","end_day","end_time","run_time",
    "boundary_run_time","okng_seq","test_contents","test_time","from_to_test_ct",
    "boundary_test_ct","diff_ct","problem1","problem2","problem3", "problem4"
]
missing = [c for c in OUT_COLS_X8 if c not in df_top.columns]
if missing:
    raise KeyError(f"[ERROR] X8 출력에 필요한 컬럼이 없습니다: {missing}")

df_top = df_top[OUT_COLS_X8].copy()

print(f"[OK] 대표행(바코드별 min okng_seq 우선) 기준 diff_ct TOP5% cut={cut95:.3f} / rows={len(df_top)}")

display_scroll_df(
    df_top,
    n=300,
    height_px=560,
    float_format={
        "run_time":"{:.2f}",
        "boundary_run_time":"{:.2f}",
        "boundary_test_ct":"{:.2f}",
        "from_to_test_ct":"{:.3f}",
        "diff_ct":"{:.3f}",
    },
    title="Cell X8: diff_ct TOP 5% (per barcode: min okng_seq -> max diff_ct, sorted by run_time desc)"
)

[OK] 대표행(바코드별 min okng_seq 우선) 기준 diff_ct TOP5% cut=-0.111 / rows=1


,group,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,okng_seq,test_contents,test_time,from_to_test_ct,boundary_test_ct,diff_ct,problem1,problem2,problem3,problem4
0,5,BA1WJ25353500062UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,20:12:31,38.90,26.33,1,0.00_d_sig_val_090_set,20:11:54.76,2.490,2.58,-0.090,relay_board,None,None,None


In [43]:
# ============================================
# Cell X9) df_top(X8 출력) 키(barcode_information,end_day,end_time)로
#          c1_fct_detail.fct_detail 매칭 → file_path 컬럼 추가
#  - 결과: df_top_fp
# ============================================

import pandas as pd
from sqlalchemy import text

# --- X8 출력 DF(df_top) 존재 확인 ---
if "df_top" not in globals() or len(df_top) == 0:
    raise ValueError("[ERROR] df_top이 없거나 비어있습니다. Cell X8을 먼저 실행하세요.")

# --- 키 정규화 유틸 ---
def norm_end_day(x):
    """
    end_day 정규화:
    - 'YYYYMMDD' -> 'YYYY-MM-DD'
    - 이미 날짜형/문자열이면 to_datetime으로 통일
    """
    s = str(x).strip()
    if s.isdigit() and len(s) == 8:
        return pd.to_datetime(s, format="%Y%m%d", errors="coerce").strftime("%Y-%m-%d")
    return pd.to_datetime(s, errors="coerce").strftime("%Y-%m-%d")

def norm_end_time(x):
    """
    end_time 정규화:
    - 숫자/문자 혼재 가능
    - '185750' 같은 HHMMSS는 6자리 유지
    - '1857' 등도 들어오면 가능한 범위에서 zfill
    """
    if pd.isna(x):
        return None
    s = str(x).strip()
    # 소수/지수표기 등 방지: 정수처럼 보이면 정수화 후 문자열
    if s.replace(".", "", 1).isdigit():
        try:
            s2 = str(int(float(s)))
            s = s2
        except Exception:
            pass
    # HHMMSS는 보통 6자리이므로 6자리로 맞춤(짧으면 0-padding)
    if s.isdigit() and len(s) < 6:
        s = s.zfill(6)
    return s

# --- df_top에서 키만 추출 (X8 출력 기준) ---
key_df = df_top[["barcode_information", "end_day", "end_time"]].copy()
key_df["barcode_information"] = key_df["barcode_information"].astype(str).str.strip()
key_df["end_day"] = key_df["end_day"].apply(norm_end_day)
key_df["end_time"] = key_df["end_time"].apply(norm_end_time)

# 유효 키만
key_df = key_df.dropna(subset=["barcode_information", "end_day", "end_time"]).copy()

barcodes = key_df["barcode_information"].drop_duplicates().tolist()
days     = key_df["end_day"].drop_duplicates().tolist()

if len(barcodes) == 0 or len(days) == 0:
    raise ValueError("[ERROR] df_top에서 barcode/end_day/end_time 유효 키를 추출하지 못했습니다.")

print(f"[OK] X9 match keys(from df_top): barcodes={len(barcodes)}, days={len(days)}, keys={len(key_df)}")

# --- fct_detail에서 file_path 로드 ---
# 주의:
# - end_time은 text로 캐스팅되므로, 양쪽 모두 6자리 문자열로 정규화해서 매칭 안정화
SQL_FILEPATH = text("""
SELECT
    barcode_information::text AS barcode_information,
    to_char(end_day, 'YYYY-MM-DD') AS end_day,
    end_time::text AS end_time,
    file_path::text AS file_path
FROM c1_fct_detail.fct_detail
WHERE barcode_information = ANY(CAST(:barcodes AS text[]))
  AND end_day = ANY(CAST(:days AS date[]))
  AND file_path IS NOT NULL
""")

df_fp = pd.read_sql(SQL_FILEPATH, engine, params={"barcodes": barcodes, "days": days})

# --- 키 정규화(조회 결과) ---
df_fp["barcode_information"] = df_fp["barcode_information"].astype(str).str.strip()
df_fp["end_day"] = pd.to_datetime(df_fp["end_day"], errors="coerce").dt.strftime("%Y-%m-%d")
df_fp["end_time"] = df_fp["end_time"].apply(norm_end_time)
df_fp["file_path"] = df_fp["file_path"].astype(str)

# (barcode,end_day,end_time) 중복 발생 시 file_path 1개로 압축
df_fp = (
    df_fp.dropna(subset=["barcode_information", "end_day", "end_time"])
         .sort_values(["barcode_information", "end_day", "end_time"])
         .drop_duplicates(["barcode_information", "end_day", "end_time"], keep="first")
         .reset_index(drop=True)
)

# --- df_top에 병합 (X8 출력 DF에 file_path 붙이기) ---
df_top_fp = df_top.copy()
df_top_fp["barcode_information"] = df_top_fp["barcode_information"].astype(str).str.strip()
df_top_fp["end_day"] = df_top_fp["end_day"].apply(norm_end_day)
df_top_fp["end_time"] = df_top_fp["end_time"].apply(norm_end_time)

df_top_fp = df_top_fp.merge(
    df_fp[["barcode_information", "end_day", "end_time", "file_path"]],
    on=["barcode_information", "end_day", "end_time"],
    how="left"
)

print(f"[OK] df_top_fp rows={len(df_top_fp)} / file_path filled={df_top_fp['file_path'].notna().sum()}")

display_scroll_df(
    df_top_fp,
    n=300,
    height_px=560,
    float_format={
        "run_time":"{:.2f}",
        "boundary_run_time":"{:.2f}",
        "boundary_test_ct":"{:.2f}",
        "from_to_test_ct":"{:.3f}",
        "diff_ct":"{:.3f}",
    },
    title="Cell X9: df_top_fp (X8 TOP5% 대표행 +file_path)"
)

[OK] X9 match keys(from df_top): barcodes=1, days=1, keys=1
[OK] df_top_fp rows=1 / file_path filled=1


,group,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,okng_seq,test_contents,test_time,from_to_test_ct,boundary_test_ct,diff_ct,problem1,problem2,problem3,problem4,file_path
0,5,BA1WJ25353500062UPC3T-14F014-AC,FCT1,Non-PD,2025-12-19,20:12:31,38.90,26.33,1,0.00_d_sig_val_090_set,20:11:54.76,2.490,2.58,-0.090,relay_board,None,None,None,\\192.168.108.155\FCT LogFile\Machine Log\FCT\2025\12\19\FORD A+C PD NONE 35643010\BA1WJ25353500062UPC3T-14F014-AC_20251219_201151.txt


In [45]:
# ============================================
# Cell X10) df_top_fp(X9 출력) 를 e4_predictive_maintenance.non_pd_worst 저장 (FIXED)
#  - 테이블 없으면 생성
#  - UPSERT(중복 시 업데이트)
#  - PK: (barcode_information, end_day, end_time, test_contents, okng_seq)
#  - ✅ group 바인드 에러 해결: group_no로 완전 치환
# ============================================

import pandas as pd
from sqlalchemy import text

# ✅ 저장 대상: X9 출력 DF (df_top_fp)
if "df_top_fp" not in globals() or len(df_top_fp) == 0:
    raise ValueError("[ERROR] df_top_fp가 없거나 비어있습니다. Cell X9를 먼저 실행하세요.")

TARGET_SCHEMA = "e4_predictive_maintenance"
TARGET_TABLE  = "non_pd_worst"
FULL_NAME     = f"{TARGET_SCHEMA}.{TARGET_TABLE}"

# -----------------------------
# 1) 저장 전 DF 정리
# -----------------------------
df_save = df_top_fp.copy()

# group -> group_no 로 저장 컬럼 확정 (없으면 NULL)
if "group" in df_save.columns and "group_no" not in df_save.columns:
    df_save["group_no"] = pd.to_numeric(df_save["group"], errors="coerce").astype("Int64")
elif "group_no" not in df_save.columns:
    df_save["group_no"] = pd.NA

# 필수키 컬럼 존재 확인
pk_cols = ["barcode_information", "end_day", "end_time", "test_contents", "okng_seq"]
missing_pk = [c for c in pk_cols if c not in df_save.columns]
if missing_pk:
    raise KeyError(f"[ERROR] PK에 필요한 컬럼이 없습니다: {missing_pk}")

# 타입 정규화
df_save["barcode_information"] = df_save["barcode_information"].astype(str).str.strip()
df_save["end_day"] = pd.to_datetime(df_save["end_day"], errors="coerce").dt.date
df_save["end_time"] = df_save["end_time"].astype(str).str.strip()
df_save["test_contents"] = df_save["test_contents"].astype(str).str.strip()
df_save["okng_seq"] = pd.to_numeric(df_save["okng_seq"], errors="coerce").astype("Int64")

# 숫자형
for c in ["run_time","boundary_run_time","from_to_test_ct","boundary_test_ct","diff_ct"]:
    if c in df_save.columns:
        df_save[c] = pd.to_numeric(df_save[c], errors="coerce")

# problem 컬럼 보장
for c in ["problem1","problem2","problem3","problem4"]:
    if c not in df_save.columns:
        df_save[c] = pd.NA

# file_path 보장
if "file_path" not in df_save.columns:
    df_save["file_path"] = pd.NA

# -----------------------------
# 2) 테이블 생성(없으면)
# -----------------------------
DDL = text(f"""
CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA};

CREATE TABLE IF NOT EXISTS {FULL_NAME} (
    group_no            integer,
    barcode_information text NOT NULL,
    station             text,
    remark              text,
    end_day             date NOT NULL,
    end_time            text NOT NULL,
    run_time            double precision,
    boundary_run_time   double precision,
    okng_seq            integer NOT NULL,
    test_contents       text NOT NULL,
    test_time           text,
    from_to_test_ct     double precision,
    boundary_test_ct    double precision,
    diff_ct             double precision,
    problem1            text,
    problem2            text,
    problem3            text,
    problem4            text,
    file_path           text,
    updated_at          timestamp without time zone DEFAULT now(),
    PRIMARY KEY (barcode_information, end_day, end_time, test_contents, okng_seq)
);
""")

# -----------------------------
# 3) UPSERT SQL (✅ 바인드도 group_no로)
# -----------------------------
save_cols = [
    "group_no",
    "barcode_information","station","remark","end_day","end_time",
    "run_time","boundary_run_time",
    "okng_seq","test_contents","test_time",
    "from_to_test_ct","boundary_test_ct","diff_ct",
    "problem1","problem2","problem3","problem4",
    "file_path"
]
# 실제 존재 컬럼만
save_cols = [c for c in save_cols if c in df_save.columns]

insert_cols_sql = ", ".join(save_cols)
values_sql      = ", ".join([f":{c}" for c in save_cols])

conflict_cols = "barcode_information, end_day, end_time, test_contents, okng_seq"
set_cols = [c for c in save_cols if c not in pk_cols]  # 키 제외 업데이트
set_sql  = ", ".join([f"{c} = EXCLUDED.{c}" for c in set_cols] + ["updated_at = now()"])

UPSERT_SQL = text(f"""
INSERT INTO {FULL_NAME} ({insert_cols_sql})
VALUES ({values_sql})
ON CONFLICT ({conflict_cols})
DO UPDATE SET {set_sql};
""")

# -----------------------------
# 4) 실행
# -----------------------------
with engine.begin() as conn:
    conn.execute(DDL)

    rows = df_save[save_cols].to_dict(orient="records")

    # Int64/NaT/NaN -> None (바인드 안정화)
    for r in rows:
        for k, v in list(r.items()):
            if pd.isna(v):
                r[k] = None

    conn.execute(UPSERT_SQL, rows)

print(f"[OK] Saved to {FULL_NAME} (rows={len(df_save)})  # from df_top_fp(X9 output)")

[OK] Saved to e4_predictive_maintenance.non_pd_worst (rows=1)  # from df_top_fp(X9 output)
